# SmartChat BERTurk Duygu Analizi Eğitimi
Bu Jupyter Notebook (Defter), Hugging Face `transformers` kütüphanesini kullanarak `train.csv` dosyanız ile model eğitmeyi ve kendi yüklediğiniz `test.csv` dosyanız ile test edip doğruluğunu (accuracy) ölçmeyi sağlar.

**ÖNEMLİ:** Bu kodu kendi bilgisayarınızda (CPU) çalıştırırsanız eğitim saatler veya günler sürebilir. En yüksek performansı almak için bu dosyayı [Google Colab](https://colab.research.google.com/)'e yükleyerek üst menüden `Çalışma Zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU` seçeneğini ayarladıktan sonra çalıştırmalısınız.

In [1]:
!pip install transformers torch datasets evaluate emoji -q

In [2]:
import pandas as pd
import numpy as np

# Bütün veri setini olduğu gibi yüklüyoruz (Yaklaşık 1.3 Milyon)
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

print("Eğitim verisi satır sayısı: ", len(train_df))
print("Test verisi satır sayısı: ", len(test_df))


Eğitim verisi satır sayısı:  440679
Test verisi satır sayısı:  48965


In [3]:
import re
import emoji

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = emoji.replace_emoji(text, replace="")
    text = re.sub(r"[^a-zA-Z0-9çğıöşüÇĞİÖŞÜ\s.,!?]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"]  = test_df["text"].apply(clean_text)

In [4]:
# Tüm etiketleri küçük harfe çevirelim ki büyük/küçük harf sorunundan kurtulalım
train_df["label"] = train_df["label"].str.lower().str.strip()
test_df["label"] = test_df["label"].str.lower().str.strip()

# Artık hepsi küçük, sadece 3 etiket tanımlamamız yeterli
labels_map = {"positive": 2, "notr": 1, "negative": 0}

train_df["label_id"] = train_df["label"].replace(labels_map)
test_df["label_id"]  = test_df["label"].replace(labels_map)

# Eksikleri veya hatalı olanları düşür
train_df = train_df.dropna(subset=["label_id"])
test_df = test_df.dropna(subset=["label_id"])

# Boş kalanların tipini kontrol et
train_df = train_df[train_df["label_id"].isin([0, 1, 2])]
test_df = test_df[test_df["label_id"].isin([0, 1, 2])]

train_df["label_id"] = train_df["label_id"].astype(int)
test_df["label_id"] = test_df["label_id"].astype(int)


In [5]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["clean_text", "label_id"]])
test_dataset = Dataset.from_pandas(test_df[["clean_text", "label_id"]])

train_dataset = train_dataset.rename_column("label_id", "label")
test_dataset = test_dataset.rename_column("label_id", "label")

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "dbmdz/bert-base-turkish-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["clean_text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/440679 [00:00<?, ? examples/s]

Map:   0%|          | 0/48965 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="smartchat_bert_model",
    eval_strategy="epoch",
    save_strategy="epoch",      # Sadece her epoch sonunda kayıt yapar (Zırt pırt kaydetmez)
    save_total_limit=1,         # DİSK KORUMASI: En fazla 1 yedek tutar, yenisi gelince eskisini siler!
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.100078,0.104512,0.971245
2,0.074988,0.113463,0.971449
3,0.055300,0.138370,0.971061


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=82629, training_loss=0.08076037884676106, metrics={'train_runtime': 25968.9055, 'train_samples_per_second': 50.908, 'train_steps_per_second': 3.182, 'total_flos': 8.696141835821798e+16, 'train_loss': 0.08076037884676106, 'epoch': 3.0})

In [9]:
# Eğitilmiş modelini kaydet!
trainer.save_model("smartchat_berturk_duygu")
tokenizer.save_pretrained("smartchat_berturk_duygu")

print("Model başarıyla kaydedildi! Bu klasörü indirip backend'e koyabilirsin.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model başarıyla kaydedildi! Bu klasörü indirip backend'e koyabilirsin.


In [2]:
from transformers import pipeline

# Kendi eğittiğiniz modeli yerel klasörden yüklüyoruz
pipe = pipeline("sentiment-analysis", model="./smartchat_berturk_duygu", tokenizer="./smartchat_berturk_duygu")

# İstediğiniz cümleleri buraya yazıp test edebilirsiniz
zor_testler = [
    "Harika bir iş çıkardın, sayende bütün projemiz mahvoldu!", # İroni (Negatif olmalı)
    "Fena değil ama beklediğimden çok daha yavaştı.",          # Karışık (Negatif/Nötr olmalı)
    "Başta çok korkmuştum ama sonunda her şey yolunda gitti.", # Zıtlık (Pozitif olmalı)
    "Keşke biraz daha dikkatli olsaydın, üzüldüm.",            # Hafif Negatif
    "Oğlum bu ne biçim bir şey ya, şaka mı yapıyorsunuz?",     # Argo/Tepki (Negatif)
    "Tamam, bakarız o zaman.",                                # Duygusuz/Kısa (Nötr olmalı)
    "Yani fena sayılmaz aslında, neyse.",                      # Belirsiz (Nötr)
    "Allah bELANI VERSİN"
]


print("--- MODEL TEST SONUÇLARI ---\n")
for cumle in zor_testler:
    sonuc = pipe(cumle)[0]
    label = sonuc['label']
    skor = sonuc['score']
    
    # Label ID'leri (0, 1, 2) anlamlı isimlere çevirelim (Modelinizdeki sıraya göre)
    if label == "LABEL_2": d_adi = "POZİTİF"
    elif label == "LABEL_0": d_adi = "NEGATİF"
    else: d_adi = "NÖTR"
    
    print(f"Cümle: {cumle}")
    print(f"Tahmin: {d_adi} (%{skor*100:.2f} güven)\n")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

--- MODEL TEST SONUÇLARI ---

Cümle: Harika bir iş çıkardın, sayende bütün projemiz mahvoldu!
Tahmin: POZİTİF (%99.87 güven)

Cümle: Fena değil ama beklediğimden çok daha yavaştı.
Tahmin: NEGATİF (%99.02 güven)

Cümle: Başta çok korkmuştum ama sonunda her şey yolunda gitti.
Tahmin: POZİTİF (%99.97 güven)

Cümle: Keşke biraz daha dikkatli olsaydın, üzüldüm.
Tahmin: POZİTİF (%98.79 güven)

Cümle: Oğlum bu ne biçim bir şey ya, şaka mı yapıyorsunuz?
Tahmin: POZİTİF (%99.34 güven)

Cümle: Tamam, bakarız o zaman.
Tahmin: POZİTİF (%98.93 güven)

Cümle: Yani fena sayılmaz aslında, neyse.
Tahmin: NEGATİF (%88.76 güven)

Cümle: Allah bELANI VERSİN
Tahmin: NEGATİF (%96.42 güven)

